# **KeyWordSearch**

this notebook is for implementing the keyword search method including the two comon used strategies : 
- Key word search by using TF-IDF method
- Key word search by using BM25 method

In [92]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer
from rank_bm25 import BM25Okapi
import chromadb

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

## **TF-IDF strategy**

In [93]:
def _rank_cases(scores, candidate_indices, index, top_k):
    if top_k <= 0:
        raise ValueError("top_k must be a positive integer")

    best_chunks = {}
    for chunk_index in candidate_indices:
        metadata = index["metadatas"][chunk_index]
        source_id = metadata.get("source_id")
        if not source_id:
            raise ValueError(f"Chunk {index['ids'][chunk_index]} has no source_id metadata")

        score = float(scores[chunk_index])
        current = best_chunks.get(source_id)
        if current is None or score > current["score"]:
            best_chunks[source_id] = {
                "document": index["documents"][chunk_index],
                "score": score,
                "id": index["ids"][chunk_index],
                "source_id": source_id,
                "metadatas": metadata,
            }

    ranked_cases = sorted(
        best_chunks.values(),
        key=lambda result: (-result["score"], result["source_id"]),
    )
    return ranked_cases[:top_k]


def KeywordSearch_TFIDF(query: str, top_k: int, index: dict) -> list:
    query_vector = index["tfidf_vectorizer"].transform([query])
    scores = cosine_similarity(query_vector, index["tfidf_vectors"]).ravel()
    candidate_indices = [i for i, score in enumerate(scores) if score > 0]
    return _rank_cases(scores, candidate_indices, index, top_k)

## **BM25 Strategy**

In [94]:
def KeywordSearch_BM25(query: str, top_k: int, index: dict) -> list:
    query_tokens = tokenizer(query, add_special_tokens=False)["input_ids"]
    if not query_tokens:
        return []

    scores = index["bm25_index"].get_scores(query_tokens)
    unique_query_tokens = set(query_tokens)
    candidate_indices = [
        i for i, document_tokens in enumerate(index["bm25_tokens"])
        if unique_query_tokens.intersection(document_tokens)
    ]
    return _rank_cases(scores, candidate_indices, index, top_k)

# **Testing the two strategies**

In [95]:
path = r"C:\Users\aliai\Desktop\M2 I2A\RAG Agentique\Backend\Collections"

In [96]:
client = chromadb.PersistentClient(path=path)

In [97]:
collection = client.get_collection(
    name="jurisprudence"
)

In [98]:
stored = collection.get(include=["documents", "metadatas"])
keyword_documents = stored["documents"]
if not keyword_documents:
    raise ValueError("The Chroma collection contains no documents")
if not (len(keyword_documents) == len(stored["ids"]) == len(stored["metadatas"])):
    raise ValueError("Chroma returned misaligned documents, IDs, and metadata")

keyword_index = {
    "documents": keyword_documents,
    "ids": stored["ids"],
    "metadatas": stored["metadatas"],
    "tfidf_vectorizer": TfidfVectorizer(),
}
keyword_index["tfidf_vectors"] = keyword_index["tfidf_vectorizer"].fit_transform(
    keyword_documents
)
keyword_index["bm25_tokens"] = tokenizer(
    keyword_documents,
    add_special_tokens=False,
    truncation=False,
)["input_ids"]
keyword_index["bm25_index"] = BM25Okapi(
    keyword_index["bm25_tokens"],
    k1=1.25,
)

print(f"Indexes built for {len(keyword_documents)} chunks")

Indexes built for 12370 chunks


## **TF-IDF strategy**

In [99]:
tf_idf_results = KeywordSearch_TFIDF(
    query="contrat de travail",
    top_k=2,
    index=keyword_index,
)

## **BM25 strategy**

In [100]:
bm25_results = KeywordSearch_BM25(
    query="contrat de travail",
    top_k=2,
    index=keyword_index,
)

# **Evaluation of the search strategy based on searching the cases**

This evaluation compares TF-IDF and BM25 using the same queries, ranking distinct cases (deduplicated by `source_id`) rather than chunks. Each case is assigned the highest score among its chunks. The relevance labels in the file are provisional and must be manually validated.

In [101]:
import json
from pathlib import Path

candidate_paths = [
    Path.cwd() / "keyword_evaluation.json",
    Path.cwd() / "Backend" / "Retreival" / "keyword_evaluation.json",
]
file_path = next((candidate for candidate in candidate_paths if candidate.is_file()), None)
if file_path is None:
    raise FileNotFoundError(
        "keyword_evaluation.json was not found from the current working directory"
    )

with file_path.open(encoding="utf-8") as file:
    evaluation_data = json.load(file)

queries = evaluation_data["queries"]

## **Reliable Evaluation at the Case Level**

Both strategies are evaluated on the same queries and at the same cutoff values (\(k = 5\) and \(k = 10\)). Each search function already returns unique `source_id` values, ranked according to the highest score among their respective chunks.

Precision uses \(k\) as the denominator, even when fewer than \(k\) cases are retrieved. Recall is calculated relative to all annotated relevant `source_id` values. MRR and nDCG are computed at \(k = 10\), using binary relevance judgments.

The labels in `keyword_evaluation.json` are preliminary. Therefore, the metrics should be considered indicative until the relevant cases have been manually validated.

In [108]:
from math import log2
from statistics import mean

K_VALUES = (5, 10, 20, 30)


def evaluate_strategy(search_function, method_name, evaluation_queries, index):
    per_query = []
    max_k = max(K_VALUES)

    for query_data in evaluation_queries:
        relevant_ids = set(query_data["relevant_source_ids"])
        if not relevant_ids:
            raise ValueError(f"{query_data['query_id']} has no relevance labels")

        ranked_results = search_function(
            query=query_data["query"],
            top_k=max_k,
            index=index,
        )
        ranked_ids = [result["source_id"] for result in ranked_results]
        if len(ranked_ids) != len(set(ranked_ids)):
            raise ValueError(f"{method_name} returned duplicate cases")

        row = {
            "method": method_name,
            "query_id": query_data["query_id"],
            "query": query_data["query"],
            "retrieved_source_ids": ranked_ids,
        }
        relevant_ranks = [
            rank for rank, source_id in enumerate(ranked_ids, start=1)
            if source_id in relevant_ids
        ]
        row["mrr@10"] = 1 / relevant_ranks[0] if relevant_ranks else 0.0

        for k in K_VALUES:
            found = sum(source_id in relevant_ids for source_id in ranked_ids[:k])
            row[f"precision@{k}"] = found / k
            row[f"recall@{k}"] = found / len(relevant_ids)

            dcg = sum(
                1 / log2(rank + 1)
                for rank, source_id in enumerate(ranked_ids[:k], start=1)
                if source_id in relevant_ids
            )
            ideal_count = min(len(relevant_ids), k)
            ideal_dcg = sum(
                1 / log2(rank + 1)
                for rank in range(1, ideal_count + 1)
            )
            row[f"ndcg@{k}"] = dcg / ideal_dcg if ideal_dcg else 0.0

        per_query.append(row)

    summary = {"method": method_name, "queries": len(per_query)}
    for metric in (
        "precision@5", "recall@5", "ndcg@5",
        "precision@10", "recall@10", "ndcg@10", "mrr@10",
        "precision@20", "recall@20", "ndcg@20", 
        "precision@30", "recall@30", "ndcg@30", 
    ):
        summary[metric] = mean(row[metric] for row in per_query)
    return summary, per_query